In [1]:

import pandas as pd
import numpy as np
import os


os.makedirs('data', exist_ok=True)

np.random.seed(42)
n_rows = 1000

wafer_ids = [f" WF-{i:04d} " if i % 3 == 0 else f"WF-{i:04d}" for i in range(1001, 1001 + n_rows)]
lot_codes = np.random.choice(['l-a1', 'L-A1 ', 'L-B2', 'l-b2 ', 'L-C3', 'L-C3 '], size=n_rows)
thickness = np.random.normal(loc=725.0, scale=3.5, size=n_rows)
thickness[np.random.choice(n_rows, size=int(n_rows * 0.05), replace=False)] = np.nan

date_formats = ['2026-08-{:02d} ', ' {:02d}/08/2026', '2026/08/{:02d}']
dates = [date_formats[i % 3].format(np.random.randint(1, 29)) for i in range(n_rows)]

df_wafers_raw = pd.DataFrame({
    'Wafer_ID': wafer_ids,
    'Lot_Code': lot_codes,
    'Thickness_nm': thickness,
    'Inspection_Date': dates
})

yield_wafer_ids = [f"WF-{i:04d}" for i in range(1001, 1001 + 800)]
defects = np.random.choice(['NONE', 'none', ' edge_scratch ', 'EDGE_SCRATCH', 'RING_DEFECT', 'ring_defect'], size=800, p=[0.5, 0.1, 0.1, 0.1, 0.1, 0.1])
passed_dies = np.random.randint(750, 1000, size=800)

df_yield_raw = pd.DataFrame({
    'Wafer_ID': yield_wafer_ids,
    'Total_Dies': 1000,
    'Passed_Dies': passed_dies,
    'Defect_Category': defects
})


df_wafers_raw.to_csv('data/raw_wafers_log.csv', index=False)
df_yield_raw.to_csv('data/raw_yield_log.csv', index=False)

print("Export CSV Files to data/ folder successfully!")

Export CSV Files to data/ folder successfully!


In [2]:

df_wafers = pd.read_csv('data/raw_wafers_log.csv')

df_wafers['Wafer_ID'] = df_wafers['Wafer_ID'].str.strip() 
df_wafers['Lot_Code'] = df_wafers['Lot_Code'].str.strip().str.upper()
df_wafers['Inspection_Date'] = pd.to_datetime(df_wafers['Inspection_Date'].str.strip(), format='mixed')

thickness_mean = df_wafers['Thickness_nm'].mean()
df_wafers['Thickness_nm'] = df_wafers['Thickness_nm'].fillna(thickness_mean)

df_wafers.head()

,Wafer_ID,Lot_Code,Thickness_nm,Inspection_Date
0,WF-1001,L-B2,722.722751,2026-08-13
1,WF-1002,L-C3,723.295061,2026-04-08
2,WF-1003,L-B2,722.926621,2026-08-02
3,WF-1004,L-C3,721.976032,2026-08-16
4,WF-1005,L-C3,725.169826,2026-09-08


In [ ]:

df_yield = pd.read_csv('data/raw_yield_log.csv')

df_master = pd.merge(df_wafers, df_yield, on='Wafer_ID', how='left')

df_master['Defect_Category'] = df_master['Defect_Category'].str.strip().str.upper()
df_master['Total_Dies'] = df_master['Total_Dies'].fillna(1000)
df_master['Passed_Dies'] = df_master['Passed_Dies'].fillna(0)
df_master['Defect_Category'] = df_master['Defect_Category'].fillna('UNINSPECTED')
df_master['Yield_Rate_%'] = (df_master['Passed_Dies'] / df_master['Total_Dies']) * 100

df_master.head()

,Wafer_ID,Lot_Code,Thickness_nm,Inspection_Date,Total_Dies,Passed_Dies,Defect_Category,Yield_Rate_%
0,WF-1001,L-B2,722.722751,2026-08-13,1000.0,867.0,NONE,86.7
1,WF-1002,L-C3,723.295061,2026-04-08,1000.0,970.0,NONE,97.0
2,WF-1003,L-B2,722.926621,2026-08-02,1000.0,849.0,EDGE_SCRATCH,84.9
3,WF-1004,L-C3,721.976032,2026-08-16,1000.0,771.0,RING_DEFECT,77.1
4,WF-1005,L-C3,725.169826,2026-09-08,1000.0,929.0,NONE,92.9


In [4]:

df_pivot = pd.pivot_table(
    df_master, 
    index='Lot_Code', 
    columns='Defect_Category',
    values='Yield_Rate_%', 
    aggfunc='mean', 
    fill_value=0
)

df_pivot

Defect_Category,EDGE_SCRATCH,NONE,RING_DEFECT,UNINSPECTED
Lot_Code,,,,
L-A1,87.886364,87.994156,86.506000,0.0
L-B2,87.109091,87.982530,86.971429,0.0
L-C3,87.266667,87.247742,86.370588,0.0
